# Global Discrimination: Optimal, Greedy, and VND

This notebook compares exhaustive Optimal allocation, Greedy allocation, and deterministic variable-neighborhood descent (VND) on independent global-discrimination benchmarks. All three methods minimize the actual equal-prior PGM error computed from the output-state Gram matrix for product, local-GHZ, and global-GHZ inputs at an absolute sensor budget $N$. The Gram matrices use consistent, unwrapped $\delta_{ijm}=c(f_i(x_m)-f_j(x_m))$ values.

The notebook plots the absolute actual PGM errors and analytical upper bounds for all three methods. It also reports paired `Greedy - Optimal` and `VND - Optimal` differences for both metrics. Actual-error gaps are nonnegative because Optimal now minimizes actual PGM error; upper-bound differences may be signed because the bound is only a secondary reporting metric.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Support launching Jupyter from either the repository root or this folder.
working_directory = Path.cwd().resolve()
MODULE_DIRECTORY = (
    working_directory / "min-error-global"
    if (working_directory / "min-error-global").is_dir()
    else working_directory
)
if str(MODULE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(MODULE_DIRECTORY))

# Reload dependencies first so rerunning this cell picks up edited algorithms.
importlib.invalidate_caches()
for module_name in (
    "input_data",
    "objective",
    "algorithms.greedy",
    "algorithms.optimal",
    "algorithms.vnd",
):
    module = importlib.import_module(module_name)
    importlib.reload(module)
comparison_analysis = importlib.import_module("comparison_analysis")
comparison_analysis = importlib.reload(comparison_analysis)
bound_curve_summary = comparison_analysis.bound_curve_summary
load_paired_results = comparison_analysis.load_paired_results
make_combined_comparison_plots = comparison_analysis.make_combined_comparison_plots
paired_summary = comparison_analysis.paired_summary
run_paired_dataset = comparison_analysis.run_paired_dataset
STATE_MODEL_LABELS = comparison_analysis.STATE_MODEL_LABELS

## Run settings

Set `RUN_ALL_CASES = False` for a reproducible test sample of exactly 100 problem instances from each compact dataset. Set it to `True` to run all 2,520 instances from each dataset. Every selected instance is evaluated by all three algorithms at each absolute $N$ in `SENSOR_BUDGETS` and for all three input-state models.

In [ ]:
DATASETS = ["compact_benchmark_1d", "compact_benchmark_2d"]

# False runs the 100-case test sample; True runs every available case.
RUN_ALL_CASES = True
RUN_MODE = "full" if RUN_ALL_CASES else "test"
TEST_CASES_PER_DATASET = 100
SENSOR_BUDGETS = tuple(range(1, 11))
VND_IMPROVEMENT_THRESHOLD = 1e-3
SAMPLE_SEED = 20260827
PROGRESS_EVERY = 10 if RUN_MODE == "test" else 50

RESULTS_ROOT = MODULE_DIRECTORY / "results" / "actual_pgm_optimal_greedy_vnd"
mode_description = (
    f"test: {TEST_CASES_PER_DATASET} cases per dataset"
    if RUN_MODE == "test"
    else "full: all 2,520 cases per dataset"
)
print("Run all cases:", RUN_ALL_CASES)
print("Run mode:", mode_description)
print("Absolute sensor budgets, N:", SENSOR_BUDGETS)
print("VND improvement threshold:", VND_IMPROVEMENT_THRESHOLD)
print("Results directory:", RESULTS_ROOT)

## Run all three algorithms

Optimal, Greedy, and VND receive the same unwrapped `delta_ijm` table, sensor budget, and input-state model for every paired run.

In [ ]:
records_by_dataset = {}
for dataset in DATASETS:
    display(Markdown(f"### Running `{dataset}`"))
    records_by_dataset[dataset] = run_paired_dataset(
        dataset,
        sensor_budgets=SENSOR_BUDGETS,
        vnd_improvement_threshold=VND_IMPROVEMENT_THRESHOLD,
        run_mode=RUN_MODE,
        test_case_count=TEST_CASES_PER_DATASET,
        sample_seed=SAMPLE_SEED,
        progress_every=PROGRESS_EVERY,
        results_root=RESULTS_ROOT,
    )

## Overall paired-gap summary

Each difference is `method - Optimal`, where Optimal is the exhaustive minimum of actual Gram-matrix PGM error. Actual-error gaps are therefore nonnegative. Upper-bound differences are signed because none of the methods optimize the analytical bound.

In [ ]:
summary_lines = [
    "| Dimension | Input state | Cases | Runs | Greedy bound difference | VND bound difference | Greedy PGM-error gap | VND PGM-error gap |",
    "|---|---|---:|---:|---:|---:|---:|---:|",
]
for records in records_by_dataset.values():
    for row in paired_summary(records):
        summary_lines.append(
            f"| {row['dimension']} | {STATE_MODEL_LABELS[row['state_model']]} | "
            f"{row['cases']} | {row['runs']} | "
            f"{row['mean_greedy_upper_bound_difference']:.5f} | "
            f"{row['mean_vnd_upper_bound_difference']:.5f} | "
            f"{row['mean_greedy_pgm_error_gap']:.5f} | "
            f"{row['mean_vnd_pgm_error_gap']:.5f} |"
        )
display(Markdown("\n".join(summary_lines)))

## Paired gaps by sensor budget

In [ ]:
curve_lines = [
    r"| Dimension | Input state | Sensor budget $N$ | Greedy bound difference | VND bound difference | Greedy PGM-error gap | VND PGM-error gap | Runs |",
    "|---|---|---:|---:|---:|---:|---:|---:|",
]
for records in records_by_dataset.values():
    for row in bound_curve_summary(records):
        curve_lines.append(
            f"| {row['dimension']} | {STATE_MODEL_LABELS[row['state_model']]} | "
            f"{row['sensor_budget']} | "
            f"{row['mean_greedy_upper_bound_difference']:.5f} | "
            f"{row['mean_vnd_upper_bound_difference']:.5f} | "
            f"{row['mean_greedy_pgm_error_gap']:.5f} | "
            f"{row['mean_vnd_pgm_error_gap']:.5f} | "
            f"{row['runs']} |"
        )
display(Markdown("\n".join(curve_lines)))

## Three-method and paired-gap plots

Four figures are produced. The first two show absolute analytical upper bounds and actual PGM errors for Optimal, Greedy, and VND. The remaining figures preserve the `method - Optimal` bound-difference and actual-error-gap views. Distinct markers make coincident method means visible.

In [ ]:
plt.close("all")

# Load saved records when the expensive run cell was skipped intentionally.
if "records_by_dataset" not in globals():
    records_by_dataset = {
        dataset: load_paired_results(
            dataset, results_root=RESULTS_ROOT, run_mode=RUN_MODE
        )
        for dataset in DATASETS
    }

for title, figure in make_combined_comparison_plots(
    records_by_dataset, results_root=RESULTS_ROOT, run_mode=RUN_MODE
):
    display(Markdown(f"## {title}"))
    display(figure)
    plt.close(figure)

## Saved files

Detailed JSON and CSV tables are saved separately for each dimension. Two absolute comparison figures and two Optimal-relative figures are saved as 300-dpi PNG and vector PDF files.

```text
min-error-global/results/actual_pgm_optimal_greedy_vnd/
├── compact_benchmark_1d_results_<test|full>/tables/
├── compact_benchmark_2d_results_<test|full>/tables/
└── results_summary_compact_benchmark_1d_compact_benchmark_2d_<test|full>/figures/
```